# MIRI Drug Repositioning Screen — xpert-fda-v2 (HG-aligned)

Single orchestrator for the numbered workflow steps (01-11). Run cells in order. Each step lives in its own directory with its script, `output/` data, and plots, and reads its inputs from previous steps (plus static files in `input/`).

This workflow keeps the L1000 HG rows byte-identical to the author reference and embeds the FDA-extension drugs with author-pinned frozen inference (see `docs/adr/0006`); `input/` is a symlink to v1's `input/`.

Environment: `conda activate arc` (see root `environment.yml`). Heavy steps (02 embeddings, 04 finetune, 05 benchmark, 06 predict) require a GPU.

## Step 01 — Drug list

Generates the 3,127-drug FDA list from the ChEMBL source table (`input/fda_drugs.csv`). Output: `01_drug_list/output/drug_list.csv`.

In [ ]:
!python 01_drug_list/01_drug_list.py

## Step 02 — Drug embeddings (HG-aligned)

Verifies the frozen UniMol array + SMILES mapping (`input/`), recomputes the FDA DDS edges with `Tanimoto > 0.5`, and generates the HG biological embeddings via author-pinned frozen inference (rows 0-8980 = author reference byte-identical by construction). Idempotent: skips inference when an intact `02_drug_embeddings/output/all_drugs_idx2HG.npy` passes all gates (`_lib/verify_gates.py`, gates A-D).

In [ ]:
!python 02_drug_embeddings/02_drug_embeddings.py

# Force regeneration:
# !python 02_drug_embeddings/02_drug_embeddings.py --force

## Step 03 — Baseline transcriptome

Builds the workflow-owned consensus cardiomyocyte baseline from DTOXS control wells. Idempotent. Output: `03_baseline_transcriptome/output/cm_ctl_landmark.csv`.

In [ ]:
!python 03_baseline_transcriptome/03_baseline_transcriptome.py

## Step 04 — Finetune XPert on DTOXS

Full-DToxS finetune -> the final prediction model. Idempotent (weights kept). Output: `04_finetune_xpert/output/dtoxs_dtoxs/`.

In [ ]:
!python 04_finetune_xpert/04_finetune_xpert.py

## Step 05 — Benchmark XPert on DTOXS splits

Finetunes on the warm-start / cold-drug / cold-cate splits (weights kept) and evaluates each fold on its held-out test set. Side branch: downstream steps use the step-04 model, not the benchmark weights. Outputs: `05_benchmark_xpert/output/benchmark.csv` + `benchmark.png`.

In [ ]:
!python 05_benchmark_xpert/05_benchmark_xpert.py

## Step 06 — Perturbed transcriptome

Predicts drug perturbation (deg) for all FDA drugs via XPert (step-04 model). Predict indexes the precomputed UniMol array by drug index, so finetune and predict consume identical features. Output: `06_perturbed_transcriptome/output/xpert_fda-cm_deg.csv`.

In [ ]:
!python 06_perturbed_transcriptome/06_perturbed_transcriptome.py

## Step 07 — Z-score

Z-score normalizes the predicted DEG per gene across drugs. Output: `07_zscore/output/xpert_fda-cm_deg-zs.csv`.

In [ ]:
!python 07_zscore/07_zscore.py

## Step 08 — MIRI reference gene sets + pathways

Consumes the per-GSE DEG lists in `disease_data/miri/{GSE}/deg/` and produces the integrated reference (`MIRI_up-down.json`, `MIRI_rank.json`). `input/MIRI_pathways.json` is static (curated).

In [ ]:
!python 08_miri_ref_path/08_miri_ref_path.py

## Step 09 — Efficacy scores

Runs the three scorers (WCS+RGES, Zhang/ssCMap, pathway RGES) and computes the z-components for all drugs, plus the score-distribution plots (histograms, stratified histograms, cross-method Spearman heatmap). Output: `09_efficacy_scores/output/scores.csv` + 4 PNGs in the step dir.

In [ ]:
!python 09_efficacy_scores/09_efficacy_scores.py

## Step 10 — Filter

Applies the WCS<0 and >=6/8 critical-pathway filters + filter-progression pyramid plot. Outputs: `10_filter/output/wcs_filtered.csv`, `path_filtered.csv`, `pyramid-filters.png`.

In [ ]:
!python 10_filter/10_filter.py

## Step 11 — Integrate

Computes the integrated score over ALL drugs, ranks the filtered candidates, and writes the final report: `11_integrate/output/integrated_scores.csv`, `ranked_drugs.csv`, `report.md` + score-percentile rank heatmaps (`heatmap_rank.png`, `heatmap_rank@50.png`).

In [ ]:
!python 11_integrate/11_integrate.py

## Pipeline summary

The final ranked drug list is `11_integrate/output/ranked_drugs.csv` with the deterministic report at `11_integrate/output/report.md`. The HG-aligned FDA rows diverge from v1's FDA rows by design; compare the ranking against `miri_reposition/xpert-fda-v1/11_integrate/output/ranked_drugs.csv` to review the impact of alignment. Gate harness: `python _lib/verify_gates.py`.

In [ ]:
import pandas as pd
ranked = pd.read_csv('11_integrate/output/ranked_drugs.csv')
print(f'{len(ranked)} drugs passed the filters')
print(ranked[['drug_name', 'integrated']].head(10).to_string(index=False))